# Dataset overview

Loads local processed tables from the frozen `dataset_v0.2_candidate` snapshot (no API).

In [ ]:
from pathlib import Path

import polars as pl

ROOT = Path('..').resolve()
for base in (ROOT / 'dataset_v0.2_candidate' / 'processed', ROOT / 'data' / 'processed'):
    if (base / 'works.parquet').exists():
        PROCESSED = base
        break
else:
    raise FileNotFoundError('No processed tables found')

works = pl.read_parquet(PROCESSED / 'works.parquet')
cby = pl.read_parquet(PROCESSED / 'citations_by_year.parquet')
persistence = pl.read_parquet(PROCESSED / 'persistence.parquet')

print('processed dir:', PROCESSED)
print('works:', works.height)
print('citations_by_year:', cby.height)
print('persistence:', persistence.height)

In [ ]:
print('Sector distribution')
display(works.group_by('concept_group').len().sort('len', descending=True))

year_col = 'year' if 'year' in works.columns else 'publication_year'
print('Publication year distribution')
display(works.group_by(year_col).len().sort(year_col))

In [ ]:
summary = {
    'citation_stop_censoring_share': float(persistence['censoring_flag'].mean()),
    'influence_decay_events': int(persistence['influence_decay_event'].sum()),
    'influence_decay_censoring_share': float(persistence['influence_decay_censored'].mean()),
}
print('Censoring / decay summary (HNDL cohort)')
for k, v in summary.items():
    print(f'  {k}: {v}')

print('\nGuardrail: K(t) citation persistence is not trade secret value V(t).')